In [57]:
# synth_pnc_claims_driftable.py
# Minimal, realistic, driftable synthetic claims dataset for Arize demos.
# - ~10 features (FNOL-time), includes a date column
# - Target: loss_amount (dense, no nulls)
# - Scenarios: 'good', 'data_drift', 'concept_drift', 'performance_drift'
# - Provides pred_model (frozen baseline model prediction) for side-by-side monitoring

import numpy as np
import pandas as pd

def _collision_severity(collision_type):
    # Encoded severity from type (kept simple and monotonic)
    mapping = {
        "RearEnd": 1.2, "Angle": 1.8, "Sideswipe": 1.0,
        "HeadOn": 3.2, "SingleVehicle": 1.4, "Parked": 0.4, "Other": 1.1
    }
    return np.vectorize(mapping.get)(collision_type)

def _insurance_band_score(band):
    # Map A(best)→E(worst) into numeric riskiness
    mapping = {"A_VeryGood": 0, "B_Good": 1, "C_Average": 2, "D_BelowAvg": 3, "E_Poor": 4}
    return np.vectorize(mapping.get)(band)

def _urbanicity_code(u):
    return np.vectorize({"Urban": 2, "Suburban": 1, "Rural": 0}.get)(u)

def _vehicle_type_code(v):
    return np.vectorize({"Economy":0,"Sedan":1,"SUV":2,"Pickup":3,"Luxury":4}.get)(v)

def _baseline_weights():
    # Baseline "true" relationship used to generate loss_amount in 'good'
    # Tuned to keep strong correlation without being trivial
    return dict(
        intercept = 500.0,
        w_vehicle_acv = 0.06,           # larger ACV -> higher PD severity
        w_collision_sev = 1800.0,       # collision severity multiplier
        w_med_index = 1200.0,           # medical environment
        w_driver_age = -8.0,            # younger → riskier (nonlinearity handled below too)
        w_ins_score = 350.0,            # worse band → higher loss
        w_urbanicity = 280.0,           # urban > suburban > rural
        w_report_lag = 9.0,             # more lag → slightly higher
        w_vehicle_age = 25.0,           # older car → slightly cheaper parts… but also more damage risk; keep small
        w_weather_risk = 700.0,         # bad CAT index → more loss
    )

def _concept_drift_weights():
    # Change the relationship materially (what matters changes)
    return dict(
        intercept = 700.0,
        w_vehicle_acv = 0.03,           # ACV matters less
        w_collision_sev = 2400.0,       # severity matters more
        w_med_index = 1800.0,           # medical environment dominates
        w_driver_age = -4.0,            # age effect weaker
        w_ins_score = 150.0,            # credit/score effect muted
        w_urbanicity = 120.0,           # urbanicity less predictive
        w_report_lag = 16.0,            # lag matters more
        w_vehicle_age = 10.0,
        w_weather_risk = 1200.0,        # CAT risk strongly upweighted
    )

def _frozen_model_prediction(X, w):
    # X is a dict of feature arrays (already numeric)
    # Deterministic “frozen model” = baseline linear + small calibrated link
    z = (
        w["intercept"]
        + w["w_vehicle_acv"] * X["vehicle_acv"]
        + w["w_collision_sev"] * X["collision_sev"]
        + w["w_med_index"]    * X["medical_cost_index"]
        + w["w_driver_age"]   * (X["driver_age"] - 40)
        + w["w_ins_score"]    * X["ins_score"]
        + w["w_urbanicity"]   * X["urbanicity_code"]
        + w["w_report_lag"]   * X["report_lag_days"]
        + w["w_vehicle_age"]  * X["vehicle_age_years"]
        + w["w_weather_risk"] * X["weather_cat_risk_index"]
    )
    # add a mild, smooth nonlinearity for calibration
    return np.maximum(50.0, z * (1.0 + 0.04 * X["collision_sev"]))

def _blend_weights(w0, w1, alpha):
    # 0 -> all baseline, 1 -> all concept
    return {k: (1.0 - alpha) * w0[k] + alpha * w1[k] for k in w0}


def generate_claims_scenario(
    n_rows=10000,
    id_prefix="ID",
    id_start=1,
    scenario="good",
    start_date="2024-01-01",
    months_span=12,
    seed=42,
    label_noise_sigma=None,
    mult_noise_sigma=0.0,
    hetero_strength=0.0,
    concept_strength=0.4,     # <-- NEW: 40% of the way to concept weights
    concept_nl_coef=0.02,     # <-- NEW: gentle nonlinearity
):

    rng = np.random.default_rng(seed)
    

    # ----- DATE / TIME (no deprecated 'M'/'Y' timedeltas) -----
    # Build a monthly bucket list, then sample months and add a random day (0–27)
    start_ts = pd.Timestamp(start_date)
    month_buckets = pd.period_range(start=start_ts, periods=months_span, freq='M')  # month periods
    chosen = rng.integers(0, months_span, size=n_rows)                              # which month for each row
    month_starts = month_buckets[chosen].to_timestamp(how='start')                  # first day of chosen month
    day_in_month = rng.integers(0, 28, size=n_rows)                                 # keep simple (avoid month length)
    date_of_loss = month_starts + pd.to_timedelta(day_in_month, unit='D')
    month = pd.PeriodIndex(date_of_loss, freq="M").astype(str) 

    # ----- BASE FEATURES (kept realistic but compact) -----
    # Vehicle type & ACV
    vehicle_type = rng.choice(["Economy","Sedan","SUV","Pickup","Luxury"],
                              size=n_rows, p=[0.15,0.30,0.30,0.18,0.07])
    vehicle_type_code = _vehicle_type_code(vehicle_type)

    vehicle_age_years = np.clip(rng.normal(8, 5, size=n_rows).round(), 0, 20).astype(int)
    # and in data_drift / concept_drift branches too

    base_acv_map = {"Economy":18000,"Sedan":22000,"SUV":32000,"Pickup":35000,"Luxury":55000}
    base_acv = np.vectorize(base_acv_map.get)(vehicle_type)
    # Depreciation + lognormal multiplicative noise
    vehicle_acv = base_acv * np.exp(-0.08*vehicle_age_years) * rng.lognormal(0, 0.22, size=n_rows)
    vehicle_acv = np.clip(vehicle_acv, 5000, 120000)

    # Driver & score
    driver_age = rng.integers(18, 80, size=n_rows)
    ins_band = rng.choice(["A_VeryGood","B_Good","C_Average","D_BelowAvg","E_Poor"],
                          size=n_rows, p=[0.22,0.28,0.28,0.15,0.07])
    ins_score = _insurance_band_score(ins_band)

    # Urbanicity + indices
    urbanicity = rng.choice(["Urban","Suburban","Rural"], size=n_rows, p=[0.45,0.4,0.15])
    urbanicity_code = _urbanicity_code(urbanicity)
    repair_cost_index = np.clip(
        (urbanicity=="Urban")*rng.normal(1.18,0.10,n_rows)
      + (urbanicity=="Suburban")*rng.normal(1.00,0.08,n_rows)
      + (urbanicity=="Rural")*rng.normal(0.90,0.07,n_rows), 0.75, 1.5
    )
    medical_cost_index = np.clip(
        (urbanicity=="Urban")*rng.normal(1.22,0.12,n_rows)
      + (urbanicity=="Suburban")*rng.normal(1.00,0.10,n_rows)
      + (urbanicity=="Rural")*rng.normal(0.92,0.08,n_rows), 0.75, 1.6
    )

    # Collision + weather
    collision_type = rng.choice(
        ["RearEnd","Angle","Sideswipe","HeadOn","SingleVehicle","Parked","Other"],
        size=n_rows, p=[0.32,0.22,0.12,0.03,0.22,0.06,0.03]
    )
    collision_sev = _collision_severity(collision_type)

    weather_cat_risk_index = np.clip(rng.beta(2,5,size=n_rows), 0, 1)

    # Simple operational piece
    report_lag_days = rng.choice([0,1,2,3,4,5,6,7,10,14,21,30],
                                 size=n_rows,
                                 p=[0.25,0.14,0.11,0.09,0.07,0.05,0.04,0.04,0.05,0.05,0.06,0.05])

    # ----- DRIFT SCENARIOS (feature distributions) -----

    if scenario == "data_drift":
    # ---- Strong covariate shift into poorly seen joint regions ----
        # (a) Vehicle mix & ACV tail (still marginal shift)
        vehicle_type = rng.choice(["Economy","Sedan","SUV","Pickup","Luxury"],
                                size=n_rows, p=[0.06,0.20,0.30,0.30,0.14])
        vehicle_type_code = _vehicle_type_code(vehicle_type)
        vehicle_age_years = np.clip(rng.normal(13, 7, size=n_rows).round(), 0, 25).astype(int)
        base_acv = np.vectorize(base_acv_map.get)(vehicle_type)
        vehicle_acv = base_acv * np.exp(-0.07*vehicle_age_years) * rng.lognormal(0, 0.45, size=n_rows)
        vehicle_acv = np.clip(vehicle_acv, 2500, 220000)

        # (b) Build **correlated** drivers: weather_cat_risk_index, medical_cost_index, collision_sev
        # Create 3D correlated normals with rho ~ 0.85
        rho = 0.85
        Sigma = np.array([[1, rho, rho],
                        [rho, 1, rho],
                        [rho, rho, 1]], dtype=float)
        L = np.linalg.cholesky(Sigma)
        z = rng.normal(size=(n_rows, 3)) @ L.T  # z ~ N(0, Sigma)

        # logistic-normal CDF approximation to Phi (no erf needed)
        def _phi_approx(x):
            return 1.0 / (1.0 + np.exp(-1.702 * x))  # close to standard normal CDF

        u = _phi_approx(z)  # shape (n_rows, 3)

        # (b1) Weather CAT risk = high-skewed (push to upper tail)
        weather_cat_risk_index = np.clip(u[:, 0]**0.4, 0, 1)

        # (b2) Medical cost index = high mean + spread (just use z[:,1] directly)
        medical_cost_index = np.clip(1.35 + 0.22 * z[:, 1], 0.85, 2.0)

        # (b3) Collision mix driven by u[:,2] toward severe buckets
        t = u[:, 2]
        bins = np.array([0.10, 0.43, 0.48, 0.58, 0.88, 0.94, 1.00])
        labels = np.array(["RearEnd","Angle","Sideswipe","HeadOn","SingleVehicle","Parked","Other"])
        idx = np.searchsorted(bins, t, side="right")
        collision_type = labels[idx]
        collision_sev = _collision_severity(collision_type)

        # (c) Urbanicity heavier Urban; report lags longer in high-index areas
        urbanicity = rng.choice(["Urban","Suburban","Rural"], size=n_rows, p=[0.65,0.28,0.07])
        urbanicity_code = _urbanicity_code(urbanicity)
        repair_cost_index  = np.clip((urbanicity=="Urban")*rng.normal(1.32,0.14,n_rows)
                                + (urbanicity=="Suburban")*rng.normal(1.06,0.11,n_rows)
                                + (urbanicity=="Rural")*rng.normal(0.97,0.09,n_rows), 0.80, 1.90)
        # Lag depends on med+weather (still no leakage; both are FNOL-time proxies)
        base_lag = rng.choice([0,1,2,3,4,5,6,7,10,14,21,30], size=n_rows,
                            p=[0.15,0.10,0.09,0.08,0.07,0.06,0.05,0.05,0.07,0.08,0.10,0.10])
        extra = (2.5*weather_cat_risk_index + 3.0*np.maximum(medical_cost_index-1.2, 0)).astype(int)
        report_lag_days = np.minimum(base_lag + extra, 30)

        # (d) If user didn’t override noise, add volatility so errors amplify in the joint tail
        if label_noise_sigma is None:
            label_noise_sigma = 200.0
        if mult_noise_sigma == 0.0:
            mult_noise_sigma = 0.06
        if hetero_strength == 0.0:
            hetero_strength = 0.8


    if scenario == "concept_drift":
        # Nudge distributions (not extreme, but visible in PSI/KS)
        # More pickups & luxury, older fleet, higher CAT & medical costs, longer lags
        pass

    # ----- “TRUE” LOSS GENERATION -----
    # Baseline relationship (for 'good' and 'data_drift')
    w_base = _baseline_weights()
    w_conc = _concept_drift_weights()

    # nonlinearity bump for young drivers (kept in all scenarios)
    young_bump = (driver_age < 25).astype(float) * 700.0

    def _gen_loss(weights, nl_coef=0.0):
    # Saturating ACV feature (same mapping for ALL scenarios)
        acv_feat = np.log1p(vehicle_acv / 12000.0)

        z = (
            weights["intercept"]
            + weights["w_vehicle_acv"] * acv_feat
            + weights["w_collision_sev"] * collision_sev
            + weights["w_med_index"]    * medical_cost_index
            + weights["w_driver_age"]   * (driver_age - 40)
            + weights["w_ins_score"]    * ins_score
            + weights["w_urbanicity"]   * urbanicity_code
            + weights["w_report_lag"]   * report_lag_days
            + weights["w_vehicle_age"]  * vehicle_age_years
            + weights["w_weather_risk"] * weather_cat_risk_index
            + young_bump
        )
        if nl_coef > 0.0:
            # gentler, tunable concept nonlinearity
            z = z * (1.0 + nl_coef * collision_sev * weather_cat_risk_index) \
                + (nl_coef * 600.0) * (medical_cost_index ** 2)
        return z



    if scenario == "good":
        y_true_core = _gen_loss(w_base, nl_coef=0.0)
        if label_noise_sigma is None: label_noise_sigma = 150.0

    elif scenario == "data_drift":
        y_true_core = _gen_loss(w_base, nl_coef=0.0)
        if label_noise_sigma is None: label_noise_sigma = 150.0

    elif scenario == "concept_drift":
        # same feature distributions (you already have `pass` above)
        # blend weights + mild nonlinearity
        w_blend = _blend_weights(w_base, w_conc, concept_strength)
        y_true_core = _gen_loss(w_blend, nl_coef=concept_nl_coef)
        # keep label noise similar to good so drop is from mapping change, not volatility
        if label_noise_sigma is None: label_noise_sigma = 100.0

    elif scenario == "performance_drift":
        y_true_core = _gen_loss(w_base, nl_coef=0.0)
        if label_noise_sigma is None: label_noise_sigma = 150.0
    else:
        raise ValueError("scenario must be one of: 'good', 'data_drift', 'concept_drift', 'performance_drift'")

    # noise = np.random.default_rng(seed+1).normal(0, sigma, size=n_rows)
    # if label_noise_sigma is None:
    #     if scenario == "good":            label_noise_sigma = 150.0
    #     elif scenario == "data_drift":    label_noise_sigma = 170.0
    #     elif scenario == "concept_drift": label_noise_sigma = 180.0
    #     elif scenario == "performance_drift": label_noise_sigma = 150.0

    rng_local = np.random.default_rng(seed+1)

    # --- build noise components ---
    additive = rng_local.normal(0, label_noise_sigma, size=n_rows)

    # multiplicative noise (scales with signal level)
    multiplicative = y_true_core * rng_local.normal(0, mult_noise_sigma, size=n_rows)

    # heteroskedastic noise: more dispersion for worse collisions + cost indices
    hetero_scale = hetero_strength * (0.5*collision_sev + 0.5*medical_cost_index)
    hetero = rng_local.normal(0, label_noise_sigma * hetero_scale, size=n_rows)

    total_noise = additive + multiplicative + hetero
    loss_amount = np.clip(y_true_core + total_noise, 50.0, None)  # no nulls, no negatives

    # ----- FROZEN BASELINE MODEL PREDICTION -----
    # Pretend we trained on 'good' scenario and froze these weights:
    # frozen_w = _baseline_weights()
    # pred_inputs = dict(
    #     vehicle_acv=vehicle_acv,
    #     collision_sev=collision_sev,
    #     medical_cost_index=medical_cost_index,
    #     driver_age=driver_age,
    #     ins_score=ins_score,
    #     urbanicity_code=urbanicity_code,
    #     report_lag_days=report_lag_days,
    #     vehicle_age_years=vehicle_age_years,
    #     weather_cat_risk_index=weather_cat_risk_index,
    # )
    # pred_model = _frozen_model_prediction(pred_inputs, frozen_w)

    # PERFORMANCE DRIFT: degrade predictions without changing the ground truth generation
    # if scenario == "performance_drift":
    #     # Simulate a bad model version: bias + shrinkage + extra prediction noise
    #     pred_model = (pred_model * 0.85) + 1200.0
    #     pred_model = pred_model + rng.normal(0, 600.0, size=n_rows)

    # # Clip predictions to positive
    # pred_model = np.clip(pred_model, 50.0, None)
    ids = np.arange(id_start, id_start + n_rows, dtype=int)
    ID = pd.Series([f"{id_prefix}{i:07d}" for i in ids], dtype="string")

    # ----- ASSEMBLE -----
    df = pd.DataFrame({
        "ID": ID,
        "date_of_loss": date_of_loss,                 # time axis for drift
        # Features (10-ish, numeric & compact)
        "vehicle_type_code": vehicle_type_code,       # 0..4
        "vehicle_age_years": vehicle_age_years,       # 0..20
        "vehicle_acv": np.round(vehicle_acv, 0),      # $
        "driver_age": driver_age,                     # years
        "ins_score": ins_score,                       # 0..4 (A..E)
        "urbanicity_code": urbanicity_code,           # 0..2 (Rural..Urban)
        "repair_cost_index": np.round(repair_cost_index, 3),
        "medical_cost_index": np.round(medical_cost_index, 3),
        "collision_sev": np.round(collision_sev, 2),  # ~0.4..3.2
        "weather_cat_risk_index": np.round(weather_cat_risk_index, 3),
        "report_lag_days": report_lag_days,           # 0..30
        # Labels
        "loss_amount": np.round(loss_amount, 2),
        # Model output to log to Arize (for monitoring)
        # "pred_model": np.round(pred_model, 2),
        # Useful for slicing
        "scenario": scenario,
        "month": month,
    })

    return df

if __name__ == "__main__":
    # Quick smoke test
    for sc in ["good","data_drift","concept_drift","performance_drift"]:
        d = generate_claims_scenario(5000, sc, start_date="2024-01-01", months_span=12, seed=7)
        # print(sc, d[["loss_amount","pred_model"]].corr().iloc[0,1], d.shape)
        d.to_csv(f"claims_{sc}.csv", index=False)



In [58]:
# train_eval.py
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error

FEATURES = [
    "vehicle_type_code","vehicle_age_years","vehicle_acv","driver_age","ins_score",
    "urbanicity_code","repair_cost_index","medical_cost_index","collision_sev",
    "weather_cat_risk_index","report_lag_days",
]

def train_on_good_and_eval(generate_fn, seed=7, noise_kwargs=None):
    noise_kwargs = noise_kwargs or {}

    # TRAIN on "good"
    good = generate_fn(n_rows=12000, scenario="good", seed=seed, **noise_kwargs)
    X = good[FEATURES]
    y = good["loss_amount"]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=seed)

    # Strong model (baseline)
    model = GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.06, max_depth=3, subsample=0.7, random_state=seed
    )
    model.fit(X_tr, y_tr)

    # Weak model (for performance_drift)
    bad_model = DecisionTreeRegressor(max_depth=3, random_state=seed)
    bad_model.fit(X_tr, y_tr)

    # Good metrics
    y_pred_te = model.predict(X_te)
    print("\nGOOD (held-out test) metrics:")
    print("  R²:", round(r2_score(y_te, y_pred_te), 4))
    print("  MAE:", round(mean_absolute_error(y_te, y_pred_te), 2))

    # Save preds to GOOD
    good["pred_model"] = model.predict(good[FEATURES])
    out = {"good": good}

    # Evaluate other scenarios
    for sc in ["data_drift", "concept_drift", "performance_drift"]:
        df_sc = generate_fn(n_rows=5000, scenario=sc, seed=seed+1)
        X_sc = df_sc[FEATURES]
        y_true = df_sc["loss_amount"]

        use_model = bad_model if sc == "performance_drift" else model
        y_hat = use_model.predict(X_sc)
        df_sc["pred_model"] = y_hat

        print(f"\n{sc.upper()} metrics:")
        print("  R²:", round(r2_score(y_true, y_hat), 4))
        print("  MAE:", round(mean_absolute_error(y_true, y_hat), 2))

        out[sc] = df_sc

    return model, out

if __name__ == "__main__":
    # from synth_pnc_claims_driftable import generate_claims_scenario

    # keep “good” learnable; we’ll amplify noise only in data_drift via generator defaults
    noise = dict(label_noise_sigma=160, mult_noise_sigma=0.06, hetero_strength=0.4)

    model, datasets = train_on_good_and_eval(generate_claims_scenario, seed=7, noise_kwargs=noise)
    for name, df in datasets.items():
        df.to_csv(f"claims_{name}.csv", index=False)



GOOD (held-out test) metrics:
  R²: 0.8837
  MAE: 310.35

DATA_DRIFT metrics:
  R²: 0.8238
  MAE: 458.47

CONCEPT_DRIFT metrics:
  R²: 0.7374
  MAE: 560.85

PERFORMANCE_DRIFT metrics:
  R²: 0.6601
  MAE: 511.99
